# Practice 1 · From reads to variants
### Calling SNVs and indels in a synthetic trio (GRCh38)
*Genomic Data Processing · Day 4 (10/15) · Understanding genetic variants*

> **All data in this notebook are synthetic.** The reads were simulated from the public GRCh38 reference genome. No real person's DNA is included.

**The story.** A child (**S01**) has sickle cell disease. The parents (**S02**, father; **S03**, mother) are healthy. Sickle cell disease is *recessive*: a person is affected when **both** copies of the *HBB* gene (β-globin, part of haemoglobin) carry the disease variant. A person with **one** copy is a healthy *carrier*. We sequenced the β-globin region of all three family members. Our job is to go from raw sequencing reads to a named variant, and to check every step on the way.

**What you will practise**
1. Read the input files (reference FASTA, FASTQ) and judge read quality.
2. Align reads, mark PCR duplicates and measure coverage.
3. Turn a stack of reads into genotypes (genotype likelihoods, joint calling).
4. Clean up the calls (normalisation, filtering) and measure how good they are against a known truth.
5. Use the family structure (trio analysis) and give the variant its clinical (HGVS) name.

**The pipeline**
```
FASTQ ─► QC ─► align ─► sort/index ─► mark duplicates ─► coverage ─► pileup & genotype likelihoods
      ─► joint calling ─► normalise ─► filter ─► benchmark vs truth ─► trio checks ─► HGVS name
```

**How to use this notebook.** Run the cells from top to bottom (Shift + Enter). Each step has four parts:
* **Why this step** – the biological question, in plain words.
* **Code** – the real commands. Cells that start with `%%bash` are shell (command-line) commands; the others are Python.
* **Check** – prints a few numbers and stops with an error if something looks wrong.
* **What changed? / Your turn** – a short summary and a question. Click *Answer* to check yourself.

Running everything takes about one minute on a laptop. Inputs are read from `data/`, outputs are written to `results/`.

## Step 0 · Setup and tool versions

**Why this step.** Bioinformatics tools change between versions, and so can their results (for example, a default setting of the variant caller). Writing down the versions you used is the first step of a reproducible analysis. We also create the folders for our results and define a few names we will use all the time.

*New terms:* **reproducible** = someone else can repeat the analysis and get the same numbers; **conda environment** = a folder with a fixed set of tools and versions (see `README.md`).

In [ ]:
import os, sys, re, json, gzip, math, subprocess, time
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pysam
pysam.set_verbosity(0)     # hide htslib header warnings in Python (see the note in Step 6)

# make the command-line tools of this conda environment visible to the %%bash cells
os.environ["PATH"] = os.path.dirname(sys.executable) + os.pathsep + os.environ["PATH"]

REF = "data/ref/chr11_GRCh38_betaglobin.fa.gz"   # reference genome: GRCh38 chr11 (masked, see Step 1)
REGION = "chr11:5200001-5300000"                  # the 100-kb beta-globin window that was sequenced
SAMPLES = ["S01", "S02", "S03"]
ROLE = {"S01": "child, affected", "S02": "father", "S03": "mother"}
HBB_POS = 5_227_002                               # HBB c.20A>T, the sickle cell variant

for folder in ["qc", "bam", "coverage", "vcf", "benchmark", "logs", "figures"]:
    Path("results", folder).mkdir(parents=True, exist_ok=True)

RESULTS = {}               # key numbers are collected here and saved to results/p1_results.json at the end
T_START = time.time()

def sh(cmd):
    # run a shell command and return what it prints (used in the Check cells)
    return subprocess.run(cmd, shell=True, check=True, capture_output=True, text=True).stdout

assert Path("data/ref").is_dir(), "Start Jupyter inside the practice1_variant_calling folder"
print("Working folder:", Path.cwd().name)

In [ ]:
%%bash
samtools --version | head -n 1
bcftools --version | head -n 1
minimap2 --version
fastp --version 2>&1
bgzip --version | head -n 1

In [ ]:
# Check: every tool answers, and we keep the versions
fastp_out = subprocess.run("fastp --version", shell=True, capture_output=True, text=True)
versions = {
    "samtools": sh("samtools --version").split()[1],
    "bcftools": sh("bcftools --version").split()[1],
    "htslib (bgzip/tabix)": sh("bgzip --version").split("\n")[0].split()[-1],
    "minimap2": sh("minimap2 --version").strip(),
    "fastp": (fastp_out.stdout + fastp_out.stderr).split()[-1],
    "python": sys.version.split()[0],
    "pysam": pysam.__version__, "pandas": pd.__version__, "numpy": np.__version__,
}
print(pd.Series(versions).to_string())
assert all(versions.values()), "a tool is missing - is the conda environment activated?"
RESULTS["tool_versions"] = versions

**What changed?** Nothing in the data yet. We created the `results/` folders and recorded the tool versions.

**Your turn.** A classmate runs the same commands with a different bcftools version and gets slightly different numbers. How is that possible?
<details><summary>Answer</summary>
Algorithms and default settings change between versions. Example: in the version we use, <code>bcftools mpileup</code> keeps bases down to quality 1 by default, while <code>samtools mpileup</code> ignores bases below quality 13 – and some online manuals still list 13 for bcftools. That is why we record versions and set important options (such as <code>-Q 13</code> in Step 6) explicitly.
</details>

## Step 1 · Look at the inputs

### 1a. The reference genome
**Why this step.** Reads are short pieces of DNA. To know where they come from, we compare them with a **reference genome**: one agreed DNA sequence for each chromosome, here **GRCh38** (also called hg38). Every position in the genome has an address such as `chr11:5,227,002`.

Our reference file contains the whole length of chromosome 11 (135,086,622 letters), but only the 100-kb β-globin window `chr11:5,200,001–5,300,000` contains real sequence. Every other letter is `N` (unknown). This **masking** keeps the files small, while all positions remain real GRCh38 positions – so you can compare them with genome browsers and databases.

*File types:* **FASTA** = text file of sequences (`>name` line, then the letters). **`.fai`** = index that lets tools jump to any position; **`.gzi`** = extra index needed because the file is compressed with `bgzip`.

In [ ]:
%%bash
# columns of the .fai index: sequence name and its length
cut -f 1,2 data/ref/chr11_GRCh38_betaglobin.fa.gz.fai
# one base: the sickle cell position (samtools uses 1-based positions: the first base is 1)
samtools faidx data/ref/chr11_GRCh38_betaglobin.fa.gz chr11:5227002-5227002
# 20 bases around it
samtools faidx data/ref/chr11_GRCh38_betaglobin.fa.gz chr11:5226991-5227010

In [ ]:
# Check: the reference is what we think it is.
# Note: Python (pysam) counts from 0, so position 5,227,002 is index 5,227,001.
fa = pysam.FastaFile(REF)
base = fa.fetch("chr11", HBB_POS - 1, HBB_POS)
twenty = fa.fetch("chr11", 5_226_990, 5_227_010)
window = fa.fetch("chr11", 5_200_000, 5_300_000)
outside = fa.fetch("chr11", 5_100_000, 5_100_030)
print("length of chr11:", fa.get_reference_length("chr11"))
print("base at chr11:5,227,002:", base, "| 20 bases:", twenty)
print("N letters inside the window:", window.count("N"), "| 30 letters outside:", outside)
assert fa.get_reference_length("chr11") == 135_086_622
assert base == "T" and twenty == "CAGACTTCTCCTCAGGAGTC"
assert window.count("N") == 0 and set(outside) == {"N"}
RESULTS["reference"] = {"file": REF, "chr11_length": 135_086_622, "window": REGION,
                        "base_at_5227002": base, "chr11:5226991-5227010": twenty}

**What changed?** Nothing – we only looked. We confirmed that position chr11:5,227,002 is a **T** on the reference (plus) strand.

**Your turn.** The *HBB* gene lies on the **minus** strand of chromosome 11. Which base does the *HBB* messenger RNA have at this position?
<details><summary>Answer</summary>
<b>A</b>. The minus strand is the reverse complement of the plus strand, and T pairs with A. Keep this in mind: in Step 11 the same variant is written <code>T&gt;A</code> in the VCF (plus strand) but <code>c.20A&gt;T</code> in the clinical name (transcript strand).
</details>

### 1b. The reads (FASTQ)
**Why this step.** The sequencer cut each person's DNA into **fragments** of about 380 bases and read 150 bases from **each end** of every fragment (*paired-end* reads: **R1** and **R2**, stored in two files in the same order). A FASTQ record has 4 lines: `@name`, the bases, `+`, and one **quality character** per base.

The quality is a **Phred score** *Q*: the probability that the base is wrong is $P_{error} = 10^{-Q/10}$.

| Phred *Q* | chance the base is wrong | character in our files |
|---|---|---|
| 37 | about 1 in 5,000 | `F` |
| 25 | about 1 in 300 | `:` |
| 11 | about 1 in 13 | `,` |
| 2 | about 2 in 3 (useless) | `#` |

Character → number: Phred = ASCII code − 33 (for example `F` = 70 − 33 = 37). Modern Illumina machines report only a few such "binned" values.

In [ ]:
%%bash
# the first FASTQ record of the child's R1 file
gzip -cd data/fastq/S01_R1.fastq.gz | head -n 4

In [ ]:
# Decode the first record: quality character -> Phred score -> probability of an error
with gzip.open("data/fastq/S01_R1.fastq.gz", "rt") as fh:
    name, seq, plus, qual = [next(fh).strip() for _ in range(4)]
phred = [ord(c) - 33 for c in qual]
p_error = [10 ** (-q / 10) for q in phred]
print(name, "| read length:", len(seq))
print(pd.DataFrame({"base": list(seq[:6]), "quality character": list(qual[:6]), "Phred Q": phred[:6],
                    "P(error)": [f"{p:.4f}" for p in p_error[:6]]}).to_string(index=False))
print("quality characters in this read:", dict(Counter(qual)))
print("expected number of wrong bases in this read:", round(sum(p_error), 2))

In [ ]:
# Check: R1 and R2 files must contain the same number of reads (4 lines per read)
n_reads = {}
for s in SAMPLES:
    for mate in (1, 2):
        with gzip.open(f"data/fastq/{s}_R{mate}.fastq.gz", "rt") as fh:
            n_reads[f"{s}_R{mate}"] = sum(1 for _ in fh) // 4
print(n_reads)
for s in SAMPLES:
    assert n_reads[f"{s}_R1"] == n_reads[f"{s}_R2"], f"{s}: R1 and R2 differ"
assert len(seq) == 150
RESULTS["read_pairs"] = {s: n_reads[f"{s}_R1"] for s in SAMPLES}

**What changed?** Nothing – we decoded one record. Each sample has about 10,400 read pairs (≈ 30 reads covering each base of the 100-kb window).

**Your turn.** A base has the quality character `:`. What is the chance that it is wrong?
<details><summary>Answer</summary>
<code>:</code> has ASCII code 58, so Q = 58 − 33 = 25 and P(error) = 10<sup>−2.5</sup> ≈ 0.003, about 1 in 300.
</details>

### 1c. Read quality control (QC) with fastp
**Why this step.** Before trusting the reads we check them: Are the base qualities high? Are there adapter sequences or many very short reads? `fastp` computes these statistics. Here we only ask for a **report** (no `-o`/`-O` output files, so no reads are changed).

*New terms:* **Q30 rate** = fraction of bases with Q ≥ 30; **adapter** = artificial DNA added in the lab that can be read when a fragment is shorter than the read; **cycle** = position along the read (the sequencer adds one base per cycle).

In [ ]:
%%bash
for S in S01 S02 S03; do
  fastp -i data/fastq/${S}_R1.fastq.gz -I data/fastq/${S}_R2.fastq.gz \
        -j results/qc/${S}.fastp.json -h results/qc/${S}.fastp.html -w 2 \
        2> results/logs/${S}.fastp.log
  echo "${S}: report written to results/qc/${S}.fastp.html"
done

In [ ]:
# Read the fastp reports and plot the mean quality per cycle for R1 and R2 (child, S01)
qc = {}
for s in SAMPLES:
    j = json.load(open(f"results/qc/{s}.fastp.json"))
    b = j["summary"]["before_filtering"]
    qc[s] = {"reads (R1+R2)": b["total_reads"], "Q20 %": round(100 * b["q20_rate"], 1),
             "Q30 %": round(100 * b["q30_rate"], 1), "mean length": b["read1_mean_length"],
             "GC %": round(100 * b["gc_content"], 1),
             "reads failing fastp filters": j["filtering_result"]["low_quality_reads"]
                                             + j["filtering_result"]["too_short_reads"]}
print(pd.DataFrame.from_dict(qc, orient="index").to_string())

j = json.load(open("results/qc/S01.fastp.json"))
q1 = j["read1_before_filtering"]["quality_curves"]["mean"]
q2 = j["read2_before_filtering"]["quality_curves"]["mean"]
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(range(1, 151), q1, label="R1")
ax.plot(range(1, 151), q2, label="R2")
ax.set_xlabel("position in read (cycle)")
ax.set_ylabel("mean base quality (Phred)")
ax.set_ylim(25, 40)
ax.legend()
plt.show()

In [ ]:
# Check: high overall quality, and R2 ends lower than R1 (typical for Illumina)
end_r1, end_r2 = np.mean(q1[-10:]), np.mean(q2[-10:])
print(f"mean quality of the last 10 cycles: R1 {end_r1:.1f}, R2 {end_r2:.1f}")
for s in SAMPLES:
    assert qc[s]["Q30 %"] > 85, f"{s}: unexpectedly low quality"
assert end_r2 < end_r1
RESULTS["fastp"] = qc
RESULTS["quality_curve_S01"] = {"R1_mean_by_cycle": q1, "R2_mean_by_cycle": q2}

**What changed?** No reads were changed. We learned that about 89% of bases have Q ≥ 30, quality drops towards the end of each read, R2 is a little worse than R1, and no read fails fastp's filters. Because our fragments (~380 bp) are longer than the reads (150 bp), the reads contain no adapter. We therefore **do not trim**; later tools use the base qualities to down-weight doubtful bases.

**Your turn.** Open `results/qc/S01.fastp.html`. fastp reports a duplication rate of about 0.1 %, but Step 3 will find about 4 % duplicates. Why the difference?
<details><summary>Answer</summary>
fastp estimates duplication from the read sequences alone. Two copies of the same DNA fragment are read separately on the sequencer, so they usually differ by a few sequencing errors and do not look identical letter by letter. <code>samtools markdup</code> (Step 3) instead uses where the read pair <i>maps</i> (same start and end), which recognises them.
</details>

## Step 2 · Align the reads to the reference

**Why this step.** **Alignment** (mapping) finds, for every read, the place in the reference where it matches best – allowing for a few differences, which may be sequencing errors or real variants. `minimap2` with the preset `-ax sr` (short reads) first builds an **index** of the reference (like the index of a book), then places each read pair.

We also attach a **read group** (`-R '@RG\tID:...\tSM:...'`). The `SM` field holds the **sample name**; the variant caller uses it to keep the three people apart.

The output is **SAM** text, which we convert into a compressed, **sorted** **BAM** file (reads ordered by position) and **index** (`.bai`) so tools can jump to any region.

In [ ]:
%%bash
for S in S01 S02 S03; do
  minimap2 -ax sr -t 2 -R "@RG\tID:${S}\tSM:${S}\tLB:${S}_lib1\tPL:ILLUMINA" \
      data/ref/chr11_GRCh38_betaglobin.fa.gz \
      data/fastq/${S}_R1.fastq.gz data/fastq/${S}_R2.fastq.gz \
      2> results/logs/${S}.minimap2.log \
    | samtools sort -o results/bam/${S}.sorted.bam - 2> results/logs/${S}.sort.log
  samtools index results/bam/${S}.sorted.bam
done
ls -lh results/bam/

In [ ]:
%%bash
# The BAM header: @SQ = reference sequence, @RG = read group (note SM:S01)
samtools view -H results/bam/S01.sorted.bam | grep -E "^@(SQ|RG)"
# One aligned read covering the HBB site. Columns: name, FLAG, chromosome, start position,
# mapping quality (MAPQ), CIGAR (150M = 150 bases aligned), mate chromosome (= same), mate position, fragment length
samtools view results/bam/S01.sorted.bam chr11:5227002-5227002 | head -n 1 | cut -f 1-9

In [ ]:
# Check: almost all reads should map, as proper pairs, with the expected fragment length
align = {}
for s in SAMPLES:
    fs = json.loads(sh(f"samtools flagstat -O json results/bam/{s}.sorted.bam"))["QC-passed reads"]
    stats = sh(f"samtools stats results/bam/{s}.sorted.bam | grep ^SN")
    align[s] = {"reads": fs["total"], "% mapped": fs["mapped %"],
                "% properly paired": fs["properly paired %"],
                "mean fragment length": float(re.search(r"insert size average:\s+([\d.]+)", stats).group(1))}
print(pd.DataFrame.from_dict(align, orient="index").to_string())

with pysam.AlignmentFile("results/bam/S01.sorted.bam") as bam:
    read = next(bam.fetch("chr11", HBB_POS - 1, HBB_POS))
print(f"\nread {read.query_name}: FLAG {read.flag} =", sh(f"samtools flags {read.flag}").split()[-1])
print("start", read.reference_start + 1, "| MAPQ", read.mapping_quality, "| CIGAR", read.cigarstring)

assert "SM:S01" in sh("samtools view -H results/bam/S01.sorted.bam")
for s in SAMPLES:
    assert align[s]["% mapped"] > 99 and align[s]["% properly paired"] > 98
RESULTS["alignment"] = align

**What changed?** Two FASTQ files per person (reads in random order, no position) became one sorted, indexed BAM file per person in which every read has a position, a mapping quality and a CIGAR string. All reads mapped as proper pairs, and the mean fragment length (~380 bp) matches what the lab made.

*New terms:* **FLAG** = a number whose bits describe the read (paired, mapped in a proper pair, which mate, which strand …); **MAPQ** = Phred-scaled confidence that the read is placed correctly (60 = very sure, 0 = it fits equally well elsewhere); **CIGAR** = how the read lines up (`M` aligned, `I` insertion, `D` deletion, `S` soft-clipped).

**Your turn.** Why would joint variant calling go wrong if all three BAM files had the read group `SM:sample`?
<details><summary>Answer</summary>
The caller groups reads by the SM name. With the same name everywhere it would merge the three people into one "sample" and report a single mixed genotype – the child's 1/1 and the parents' 0/1 would blur into something that looks heterozygous.
</details>

## Step 3 · Mark PCR duplicates

**Why this step.** In the lab, DNA fragments are copied by **PCR** before sequencing. Sometimes the same original fragment is sequenced several times. These **duplicates** are not independent evidence: if the original fragment carried a PCR error, every copy repeats it. Duplicates are recognised because the read pairs start and end at exactly the same positions.

The samtools recipe has four steps:
1. `collate` – put the two mates of each pair next to each other;
2. `fixmate -m` – add mate information (the `ms` and `MC` tags) that markdup needs;
3. `sort` – back to position order;
4. `markdup` – flag all but the best copy of each duplicate set with FLAG bit 1024. Reads are **marked, not deleted**, and later tools skip them.

In [ ]:
%%bash
for S in S01 S02 S03; do
  samtools collate -O -u results/bam/${S}.sorted.bam results/bam/${S}.collate_tmp \
    | samtools fixmate -m -u - - \
    | samtools sort -u - 2> results/logs/${S}.sort2.log \
    | samtools markdup -s -f results/qc/${S}.markdup_stats.txt - results/bam/${S}.markdup.bam \
      2> results/logs/${S}.markdup.log
  samtools index results/bam/${S}.markdup.bam
done
grep -E "^(READ|DUPLICATE PAIR|DUPLICATE TOTAL)" results/qc/S03.markdup_stats.txt

In [ ]:
# A planted problem: in the mother (S03) one fragment with a PCR error was copied 15 times
JACKPOT = 5_214_982        # see data/truth/planted_events.tsv (ARTIFACT_JACKPOT)

def base_counts(bam_path, pos, min_bq=13, skip_duplicates=True):
    # count the bases (A, C, G, T) that reads show at one 1-based position
    counts = Counter()
    with pysam.AlignmentFile(bam_path) as bam:
        for col in bam.pileup("chr11", pos - 1, pos, truncate=True, stepper="nofilter",
                              min_base_quality=0, ignore_overlaps=False, max_depth=10_000):
            for p in col.pileups:
                read = p.alignment
                if p.is_del or p.is_refskip or read.is_secondary or read.is_supplementary:
                    continue
                if skip_duplicates and read.is_duplicate:
                    continue
                if read.query_qualities[p.query_position] < min_bq:
                    continue
                counts[read.query_sequence[p.query_position]] += 1
    return dict(sorted(counts.items()))

before = base_counts("results/bam/S03.sorted.bam", JACKPOT, skip_duplicates=False)
after = base_counts("results/bam/S03.markdup.bam", JACKPOT)
print("S03 reads at chr11:5,214,982 (reference G) before markdup:", before, "| after:", after)

with pysam.AlignmentFile("results/bam/S03.sorted.bam") as bam:
    ends = Counter((r.reference_start + 1, r.next_reference_start + 1)
                   for r in bam.fetch("chr11", JACKPOT - 1, JACKPOT) if r.is_read1)
(start, mate_start), n_copies = ends.most_common(1)[0]
print(f"{n_copies} read pairs share read start {start:,} and mate start {mate_start:,}")

In [ ]:
# Check: duplicate rate per person, and the jackpot shrinks to one read after marking
dup = {}
for s in SAMPLES:
    fs = json.loads(sh(f"samtools flagstat -O json results/bam/{s}.markdup.bam"))["QC-passed reads"]
    dup[s] = round(100 * fs["duplicates"] / fs["total"], 2)
print("reads flagged as duplicates (%):", dup)
for s in SAMPLES:
    assert 3 < dup[s] < 6
assert before.get("C", 0) >= 10 and after.get("C", 0) <= 2 and n_copies >= 10
RESULTS["duplicates"] = {"duplicate_pct": dup, "jackpot": {
    "sample": "S03", "pos": JACKPOT, "ref": "G", "alt": "C", "copies_with_identical_ends": n_copies,
    "read_start": start, "mate_start": mate_start,
    "counts_before_markdup": before, "counts_after_markdup": after}}

**What changed?** The BAM files still contain every read, but about 4 % now carry the duplicate flag. In S03, the 15 identical copies of one fragment collapsed to a single read, so the C at chr11:5,214,982 now has 1 supporting read instead of 15.

**Your turn.** Before marking, S03 shows about 15 C and 37 G at this position. Why could a variant caller mistake this for a heterozygous variant (genotype 0/1)?
<details><summary>Answer</summary>
About 30 % of the reads show C, with high base quality. That looks like one of the two chromosome copies carrying C. Only the identical start and end positions reveal that all 15 C-reads are copies of one molecule – one piece of evidence, not fifteen. Step 9 shows the false call that appears when duplicates are not marked.
</details>

## Step 4 · Coverage (depth)

**Why this step.** **Depth** (coverage) is the number of reads covering a base. To call a heterozygous variant we need enough reads to see both alleles: at 30× we expect about 15 reads of each allele, at 8× only about 4, and by chance sometimes 1 or 0. We measure the mean depth, how much of the window reaches 20×, and look for regions with too few reads.

`samtools depth -a` reports the depth at **every** position (also zero). Like the variant caller, it skips reads flagged as duplicates.

In [ ]:
%%bash
samtools depth -a -H -r chr11:5200001-5300000 \
    results/bam/S01.markdup.bam results/bam/S02.markdup.bam results/bam/S03.markdup.bam \
    -o results/coverage/depth.tsv
head -n 3 results/coverage/depth.tsv
wc -l < results/coverage/depth.tsv

In [ ]:
depth = pd.read_csv("results/coverage/depth.tsv", sep="\t")
depth.columns = ["chrom", "pos"] + SAMPLES
LOWCOV = (5_291_001, 5_292_000)     # see data/truth/planted_events.tsv (LOWCOV_INTERVAL)
low = depth.pos.between(*LOWCOV)
cov = pd.DataFrame({"mean depth": depth[SAMPLES].mean(),
                    "% of window >= 20x": 100 * (depth[SAMPLES] >= 20).mean(),
                    "% of window < 10x": 100 * (depth[SAMPLES] < 10).mean(),
                    "mean depth in chr11:5,291,001-5,292,000": depth.loc[low, SAMPLES].mean()}).round(1)
print(cov.to_string())

genes = pd.read_csv("data/annotation/genes_window_hg38.bed", sep="\t", comment="#", header=None,
                    names=["chrom", "start", "end", "gene", "score", "strand", "ensembl", "refseq"])
bins = depth.groupby((depth.pos - 5_200_001) // 500)[SAMPLES].mean()     # 500-bp bins
x = (5_200_001 + bins.index * 500 + 250) / 1e6
fig, (ax, gx) = plt.subplots(2, 1, figsize=(11, 3.6), sharex=True, gridspec_kw={"height_ratios": [4, 1]})
for s in SAMPLES:
    ax.plot(x, bins[s], lw=1, label=f"{s} ({ROLE[s]})")
ax.axvspan(LOWCOV[0] / 1e6, LOWCOV[1] / 1e6, color="orange", alpha=0.3, label="low-coverage interval")
ax.set_ylabel("mean depth\n(500-bp bins)")
ax.legend(ncol=4, fontsize=8, loc="lower left")
for _, g in genes.iterrows():
    gx.plot([max(g.start, 5_200_000) / 1e6, g.end / 1e6], [0, 0], lw=6, color="gray")
    gx.text((max(g.start, 5_200_000) + g.end) / 2e6, 0.6, g.gene, ha="center", fontsize=8, style="italic")
gx.set_ylim(-1, 1.5); gx.set_yticks([])
gx.set_xlabel("chr11 position (Mb, GRCh38)")
plt.show()

In [ ]:
# Check: ~30x overall, and the planted interval is clearly lower
for s in SAMPLES:
    assert 25 < cov.loc[s, "mean depth"] < 35
    assert cov.loc[s, "mean depth in chr11:5,291,001-5,292,000"] < 12
print("positions with depth < 10 inside the interval:",
      {s: int((depth.loc[low, s] < 10).sum()) for s in SAMPLES}, "of", int(low.sum()))
RESULTS["coverage"] = {s: {"mean_depth": float(cov.loc[s, "mean depth"]),
                           "pct_window_ge20x": float(cov.loc[s, "% of window >= 20x"]),
                           "pct_window_lt10x": float(cov.loc[s, "% of window < 10x"]),
                           "mean_depth_lowcov_interval": float(cov.loc[s, "mean depth in chr11:5,291,001-5,292,000"])}
                       for s in SAMPLES}
RESULTS["coverage_interval"] = {"low_coverage_interval": f"chr11:{LOWCOV[0]}-{LOWCOV[1]}", "bin_size": 500}

**What changed?** Nothing in the BAM files; we produced a depth table (`results/coverage/depth.tsv`). Mean depth is about 30×, with dips at the two ends of the window (fragments cannot extend beyond it) and in the planted 1-kb interval, where depth is only about 8×.

**Your turn.** At a heterozygous site covered by 8 reads, what is the chance that **none** of them shows the variant allele? And at 30 reads?
<details><summary>Answer</summary>
Each read comes from either chromosome copy with probability ½, so P(no variant read) = 0.5<sup>8</sup> ≈ 0.4 % at 8× and 0.5<sup>30</sup> ≈ 10<sup>−9</sup> at 30×. Seeing only 1–2 variant reads at 8× is quite common (about 14 %), and a caller may then miss or under-call the variant.
</details>

## Step 5 · Pileup at *HBB* and genotype likelihoods

**Why this step.** A **pileup** lines up all reads at one position, so we can count the bases. From the counts we want a **genotype** (diploid: two copies):
* **0/0** – both copies carry the reference allele (T),
* **0/1** – one T and one A (heterozygous, *carrier*),
* **1/1** – both copies carry A (homozygous alternate, *affected*).

In the `samtools mpileup` output, each person has three columns: depth, read bases, base qualities. In the read bases, `.` and `,` = same as the reference (forward / reverse strand), `A` and `a` = an A (forward / reverse), `^` marks a read start (followed by one mapping-quality character), `$` a read end. `-B` shows the base qualities as in the FASTQ, with one exception: when both reads of a pair cover the position, samtools counts the pair once, adding the two qualities into one base (so a `k` = Q74 can appear).

In [ ]:
%%bash
samtools mpileup -B -f data/ref/chr11_GRCh38_betaglobin.fa.gz -r chr11:5227002-5227002 \
    results/bam/S01.markdup.bam results/bam/S02.markdup.bam results/bam/S03.markdup.bam

In [ ]:
def count_pileup(bases, ref):
    # count the alleles in a samtools mpileup read-base string
    counts, i = Counter(), 0
    while i < len(bases):
        c = bases[i]
        if c == "^":                       # read start, followed by a mapping-quality character
            i += 2; continue
        if c in "+-":                      # an indel such as +2CA: skip its letters
            m = re.match(r"[+-](\d+)", bases[i:])
            i += len(m.group(0)) + int(m.group(1)); continue
        if c in ".,":
            counts[ref] += 1
        elif c.upper() in "ACGT":
            counts[c.upper()] += 1
        i += 1                             # '$' (read end) and '*' (deleted base) are skipped
    return counts

fields = sh(f"samtools mpileup -B -f {REF} -r chr11:{HBB_POS}-{HBB_POS} "
            + " ".join(f"results/bam/{s}.markdup.bam" for s in SAMPLES)).rstrip("\n").split("\t")
pile = {}
for k, s in enumerate(SAMPLES):
    c = count_pileup(fields[4 + 3 * k], fields[2])
    pile[s] = {"T (ref)": c["T"], "A (alt)": c["A"], "VAF": round(c["A"] / (c["A"] + c["T"]), 2)}
print(pd.DataFrame.from_dict(pile, orient="index").to_string())

**A worked genotype likelihood.** Take the father (S02) with *n* reads, of which *k* show A. Assume each base is wrong with probability ε = 0.001 (Phred 30). For each genotype, what is the chance that one read shows A?

| genotype | chance a read shows A | why |
|---|---|---|
| 0/0 | ε | every A must be a sequencing error |
| 0/1 | ½ | the read comes from either copy |
| 1/1 | 1 − ε | every T must be an error |

The **likelihood** of the data under genotype *G* is the binomial probability $L(G) = \binom{n}{k}\,p^{k}(1-p)^{n-k}$. The factor $\binom{n}{k}$ is the same for all three genotypes, so it cancels when we compare them. Callers report **PL** (Phred-scaled likelihood): $PL(G) = -10\log_{10}\big(L(G)/L(\text{best})\big)$. The best genotype gets PL = 0; PL = 30 means "1,000 times less likely than the best".

In [ ]:
n_ref, n_alt = pile["S02"]["T (ref)"], pile["S02"]["A (alt)"]
eps = 0.001
p_alt = {"0/0": eps, "0/1": 0.5, "1/1": 1 - eps}
log10L = {g: n_alt * math.log10(p) + n_ref * math.log10(1 - p) for g, p in p_alt.items()}
best = max(log10L.values())
PL_hand = {g: round(-10 * (v - best)) for g, v in log10L.items()}

# bcftools mpileup computes the same kind of numbers (first three PL values = 0/0, 0/1, 1/1)
out = sh(f"bcftools mpileup -Q 13 -f {REF} -r chr11:{HBB_POS} -a AD,DP results/bam/S02.markdup.bam 2>/dev/null"
         " | bcftools query -f '%REF %ALT [%PL] [%AD]\n'")
ref_b, alt_b, pl_b, ad_b = out.split()
PL_bcf = dict(zip(["0/0", "0/1", "1/1"], map(int, pl_b.split(",")[:3])))
print(f"father S02: {n_ref} T reads, {n_alt} A reads (bcftools ALT column: {alt_b}, AD {ad_b})")
print(pd.DataFrame({"P(read shows A)": p_alt, "log10 L (by hand)": {g: round(v, 1) for g, v in log10L.items()},
                    "PL (by hand)": PL_hand, "PL (bcftools)": PL_bcf}).to_string())

In [ ]:
# Check: both methods choose 0/1 for the father
assert min(PL_hand, key=PL_hand.get) == "0/1"
assert min(PL_bcf, key=PL_bcf.get) == "0/1"
RESULTS["genotype_likelihood_example"] = {
    "sample": "S02", "pos": HBB_POS, "n_ref_T": n_ref, "n_alt_A": n_alt, "epsilon": eps,
    "log10_likelihood_by_hand": {g: round(v, 2) for g, v in log10L.items()},
    "PL_by_hand": PL_hand, "PL_bcftools": PL_bcf}
RESULTS["hbb_pileup"] = pile

**What changed?** Nothing in the files. We turned base counts into three numbers per person – one for each possible genotype – and saw that bcftools ranks the genotypes the same way we did by hand. bcftools' PL values for the unlikely genotypes are smaller than ours because its model is more cautious: it uses each read's own base quality, caps very high qualities, and assumes that errors at one position are not fully independent (several reads can share the same error). It also stores every PL above 255 as 255.

**Your turn.** Change `eps = 0.001` to `eps = 0.01` and re-run the cell. What happens to the PL values, and does the best genotype change?
<details><summary>Answer</summary>
PL(0/0) and PL(1/1) become smaller, because errors are now more plausible. The best genotype stays 0/1: about 40 % A reads cannot be explained by a 1 % error rate.
</details>

## Step 6 · Joint variant calling with bcftools

**Why this step.** Now we look at **every** position of the window and for **all three people together** (*joint calling*). Calling the family together helps: evidence for an allele in one person makes it easier to genotype the others correctly at the same site.

Two programs work as a pipe (`|` sends the output of one into the next):
* `bcftools mpileup` – builds the pileups and computes genotype likelihoods (PL), like Step 5.
  `-f` reference · `-r` region · `-Q 13` ignore bases with quality below 13 (we set this explicitly; the default of this bcftools version is 1) · `-a AD,DP` also write allele depths and read depth per person · `-Ou` pass the data on uncompressed.
* `bcftools call` – combines the likelihoods with a **prior** (how common variants are) and decides alleles and genotypes.
  `-m` multiallelic caller · `-v` write only variant sites · `-a GQ` add genotype quality · `--ploidy GRCh38` humans are diploid on chr11 · `-Oz -o` write a compressed VCF.

In [ ]:
%%bash
bcftools mpileup -f data/ref/chr11_GRCh38_betaglobin.fa.gz -r chr11:5200001-5300000 \
    -Q 13 -a AD,DP -Ou \
    results/bam/S01.markdup.bam results/bam/S02.markdup.bam results/bam/S03.markdup.bam \
    2> results/logs/mpileup_main.log \
  | bcftools call -m -v -a GQ --ploidy GRCh38 -Oz -o results/vcf/trio.calls.raw.vcf.gz \
    2> results/logs/call_main.log
bcftools index -f -t results/vcf/trio.calls.raw.vcf.gz
echo "variant records: $(bcftools view -H results/vcf/trio.calls.raw.vcf.gz 2>> results/logs/warnings.log | wc -l)"

> **Note.** Tools reading these VCF files may print `[W::bcf_hdr_check_sanity] MQ should be declared as Type=Float`. This harmless warning comes from htslib checking the header written by `bcftools mpileup` (it declares the mapping-quality tag `MQ` as a whole number). We send such messages to `results/logs/warnings.log` with `2>>`.

**VCF anatomy.** A **VCF** (Variant Call Format) file has header lines starting with `##` (they define every tag), one `#CHROM` line with the column names, and one line per variant site:

| column | meaning |
|---|---|
| CHROM, POS | chromosome and 1-based position (plus strand) |
| ID | database identifier (e.g. rs334); `.` = none added |
| REF, ALT | reference allele and alternative allele(s), written on the **plus** strand |
| QUAL | Phred-scaled confidence that the **site** is variant in at least one person |
| FILTER | `PASS` or the name of a failed filter (`.` = no filters applied yet) |
| INFO | site-level tags, e.g. `DP` total depth, `MQ` mean mapping quality |
| FORMAT | the order of the per-person fields, e.g. `GT:PL:DP:AD:GQ` |
| S01, S02, S03 | the values for each person |

In [ ]:
%%bash
bcftools view -h results/vcf/trio.calls.raw.vcf.gz 2>> results/logs/warnings.log | grep -E "^##FORMAT|^#CHROM"
echo "--- the HBB record:"
bcftools view -H -r chr11:5227002 results/vcf/trio.calls.raw.vcf.gz 2>> results/logs/warnings.log

In [ ]:
# The HBB record, field by field
with pysam.VariantFile("results/vcf/trio.calls.raw.vcf.gz") as vf:
    rec = next(vf.fetch("chr11", HBB_POS - 1, HBB_POS))
    print(f"CHROM {rec.chrom} | POS {rec.pos:,} | ID {rec.id or '.'} | REF {rec.ref} | ALT {','.join(rec.alts)}"
          f" | QUAL {rec.qual:.1f} | FILTER {','.join(rec.filter) or '.'}")
    print(f"INFO/DP {rec.info['DP']} reads in total | INFO/MQ {rec.info['MQ']} (mean mapping quality)")
    hbb = {}
    for s in SAMPLES:
        smp = rec.samples[s]
        hbb[s] = {"role": ROLE[s], "GT": "/".join(map(str, smp["GT"])), "DP": smp["DP"],
                  "AD (T,A)": smp["AD"], "PL (0/0,0/1,1/1)": smp["PL"], "GQ": smp["GQ"]}
    site_qual = rec.qual
print(pd.DataFrame.from_dict(hbb, orient="index").to_string())

In [ ]:
# Check: the expected family pattern at HBB c.20A>T
n_raw = int(sh("bcftools view -H results/vcf/trio.calls.raw.vcf.gz 2>/dev/null | wc -l"))
print("variant records in the raw VCF:", n_raw, "| HBB site QUAL:", round(site_qual, 1))
assert hbb["S01"]["GT"] == "1/1" and hbb["S02"]["GT"] == "0/1" and hbb["S03"]["GT"] == "0/1"
assert site_qual > 100
RESULTS["hbb_site"] = {"pos": HBB_POS, "ref": "T", "alt": "A", "QUAL": round(site_qual, 2),
                       "samples": {s: {"GT": v["GT"], "DP": v["DP"], "AD": list(v["AD (T,A)"]),
                                       "GQ": v["GQ"], "PL": list(v["PL (0/0,0/1,1/1)"])} for s, v in hbb.items()},
                       "pileup_counts": pile}
RESULTS["record_counts"] = {"raw": n_raw}

**What changed?** BAM files (reads) became one VCF file (variants) for the whole family: each line is a site, each person gets a genotype. The *HBB* site shows the expected pattern: child 1/1, both parents 0/1.

**QUAL vs GQ.** **QUAL** is about the **site**: how sure are we that *anybody* in the family carries the A allele? (QUAL ≈ 700 means the chance that nobody does is about 10<sup>−70</sup>.) **GQ** is about **one person's genotype**: the Phred-scaled chance that this genotype is wrong (roughly the PL of the second-best genotype; bcftools caps it at 127). **PL** are the three genotype likelihoods from Step 5, **AD** the read counts per allele (REF, ALT), **DP** the reads used for this person.

**Your turn.** All of the child's reads show A, yet the child's GQ (about 74) is lower than the parents' (127). Why?
<details><summary>Answer</summary>
For the child the second-best genotype is 0/1. If the child were 0/1, seeing 25 A reads and no T would have probability 0.5<sup>25</sup> ≈ 3 × 10<sup>−8</sup>, which is Phred ≈ 75. The only way to be more certain is to have more reads. For the parents, the alternatives (0/0 or 1/1) would need many high-quality sequencing errors, which is far less likely – so their GQ reaches the cap.
</details>

## Step 7 · Normalise the variants

**Why this step.** The same DNA change can be written in different ways. Deleting one T from the run `CTTTTTTTT` gives the same sequence whichever T you delete, so a VCF could put the deletion at the start of the run or at its end. Some callers also write long REF/ALT strings or several ALT alleles in one line. Two VCF files can therefore describe identical variants and still look different to a computer.

**Normalisation** gives every variant one standard spelling: `bcftools norm -f ref` makes indels **left-aligned** (moved as far left as possible) and **parsimonious** (no unnecessary bases), and `-m -any` **splits** multi-allelic records into one line per ALT allele. Always normalise **both** files before comparing them.

In [ ]:
%%bash
bcftools norm -f data/ref/chr11_GRCh38_betaglobin.fa.gz -m -any \
    results/vcf/trio.calls.raw.vcf.gz -Oz -o results/vcf/trio.calls.norm.vcf.gz \
    2> results/logs/norm_main.log
bcftools index -f -t results/vcf/trio.calls.norm.vcf.gz
grep "Lines" results/logs/norm_main.log

In [ ]:
def records(vcf, start, end):
    # (POS, REF, ALT) of every record overlapping chr11:start-end
    with pysam.VariantFile(vcf) as vf:
        return [(r.pos, r.ref, ",".join(r.alts)) for r in vf.fetch("chr11", start - 1, end)]

RAW, NORM = "results/vcf/trio.calls.raw.vcf.gz", "results/vcf/trio.calls.norm.vcf.gz"
examples = {"1-bp deletion in a run of 8 T": (5_239_407, 5_239_415),
            "2-bp insertion in a CA repeat": (5_230_685, 5_230_699),
            "multi-allelic SNV": (5_245_200, 5_245_200)}
norm_examples = {}
for label, (a, b) in examples.items():
    norm_examples[label] = {"before": records(RAW, a, b), "after": records(NORM, a, b)}
    print(f"{label}\n   before norm: {norm_examples[label]['before']}\n   after norm:  {norm_examples[label]['after']}")
with pysam.VariantFile(NORM) as vf:
    multi_after = [{"pos": r.pos, "ref": r.ref, "alt": r.alts[0],
                    "GT": {s: "/".join(map(str, r.samples[s]["GT"])) for s in SAMPLES},
                    "AD": {s: list(r.samples[s]["AD"]) for s in SAMPLES}} for r in vf.fetch("chr11", 5_245_199, 5_245_200)]
with pysam.VariantFile(RAW) as vf:
    r = next(vf.fetch("chr11", 5_245_199, 5_245_200))
    multi_before = {"pos": r.pos, "ref": r.ref, "alt": ",".join(r.alts),
                    "GT": {s: "/".join(map(str, r.samples[s]["GT"])) for s in SAMPLES},
                    "AD": {s: list(r.samples[s]["AD"]) for s in SAMPLES}}
print("\nmulti-allelic record before splitting:", multi_before["GT"])
for m in multi_after:
    print(f"   after splitting, ALT {m['alt']}:", m["GT"])

**The truth file needs the same treatment.** `data/truth/truth_raw_unnormalised.vcf.gz` contains the planted variants, but written the "raw" way: the T-run deletion is shifted to the **right** end of the run and the multi-allelic site is **one** record. Let us compare it naively (same POS, REF and ALT text) with our calls, then normalise it and compare again.

In [ ]:
%%bash
bcftools norm -f data/ref/chr11_GRCh38_betaglobin.fa.gz -m -any \
    data/truth/truth_raw_unnormalised.vcf.gz -Oz -o results/vcf/truth_raw.norm.vcf.gz \
    2> results/logs/norm_truth.log
grep "Lines" results/logs/norm_truth.log

In [ ]:
def keys(vcf):
    # every record as (POS, REF, ALT) text - exactly what a naive comparison sees
    with pysam.VariantFile(vcf) as vf:
        return {(r.pos, r.ref, ",".join(r.alts)) for r in vf}

TRUTH_RAW = "data/truth/truth_raw_unnormalised.vcf.gz"
comparisons = {"raw calls vs raw truth": (RAW, TRUTH_RAW),
               "normalised calls vs raw truth": (NORM, TRUTH_RAW),
               "normalised calls vs normalised truth": (NORM, "results/vcf/truth_raw.norm.vcf.gz")}
rows = []
for label, (calls, truth_file) in comparisons.items():
    c, t = keys(calls), keys(truth_file)
    rows.append({"comparison": label, "shared": len(c & t), "only in calls": len(c - t), "only in truth": len(t - c)})
naive = pd.DataFrame(rows)
print(naive.to_string(index=False))
print("\nnaive mismatches (raw calls vs raw truth):")
for k in sorted(keys(RAW) ^ keys(TRUTH_RAW)):
    print("   ", k, "<- calls" if k in keys(RAW) else "<- truth")

In [ ]:
# Check: after normalising both sides, every record matches - and equals the provided truth file
n_norm = len(keys(NORM))
print("records: raw", n_raw, "-> normalised", n_norm)
assert naive.iloc[2]["only in calls"] == 0 and naive.iloc[2]["only in truth"] == 0
assert naive.iloc[0]["only in truth"] > 0
assert keys("results/vcf/truth_raw.norm.vcf.gz") == keys("data/truth/truth_trio.vcf.gz")
RESULTS["record_counts"]["normalised"] = n_norm
RESULTS["normalisation"] = {
    "examples": norm_examples,
    "multiallelic_before_split": multi_before, "multiallelic_after_split": multi_after,
    "truth_demo": naive.to_dict(orient="records"),
    "truth_raw_homopolymer_deletion": [k for k in keys(TRUTH_RAW) if k[0] == 5_239_414],
    "truth_normalised_homopolymer_deletion": [k for k in keys("data/truth/truth_trio.vcf.gz") if k[0] == 5_239_407]}

**What changed?** 33 raw records became 34: the multi-allelic site was split in two, and the two indels were rewritten in their shortest, left-most form (`CTTTTTTTT>CTTTTTTT` became `CT>C`). The naive comparison of raw files disagreed on several variants that are in fact identical; after normalising both sides, everything matched.

**VCF vs HGVS.** VCF always shifts indels to the **left** on the plus strand. The clinical HGVS nomenclature (Step 11) shifts them to the **3′ end of the transcript**. For a plus-strand gene these are opposite directions; for a minus-strand gene such as *HBB*, 3′ of the transcript is to the left on the genome, so the two rules happen to agree. Tools that convert between them must handle this.

**Your turn.** Write the deletion of one T from the run (C at 5,239,407, then eight T at 5,239,408–5,239,415) in two different but correct VCF ways.
<details><summary>Answer</summary>
Left-aligned: <code>POS=5239407 REF=CT ALT=C</code>. Right-shifted: <code>POS=5239414 REF=TT ALT=T</code>. Both produce a run of seven T – the same haplotype. <code>bcftools norm</code> turns the second form into the first.
</details>

## Step 8 · Filter the calls

**Why this step.** Not every call deserves the same trust. **Filters** use the quality numbers to flag doubtful calls. There are two levels:
* **site-level** – about the whole line, e.g. low `QUAL` or low total depth `INFO/DP`;
* **genotype-level** – about one person at one site, e.g. low `FMT/DP` or low `GQ`.

With `-s LowQual` the failing sites are **soft-filtered**: they stay in the file but their FILTER column says `LowQual` instead of `PASS`. With `-S .` failing genotypes are set to missing (`./.`). In the genotype expression a single `|` means "for the same person".

In [ ]:
%%bash
# 1) site level: mark sites with QUAL < 30 or fewer than 30 reads in the whole family
bcftools filter -s LowQual -e 'QUAL<30 || INFO/DP<30' \
    results/vcf/trio.calls.norm.vcf.gz -Oz -o results/vcf/trio.calls.sitefilt.vcf.gz \
    2>> results/logs/warnings.log
# 2) genotype level: a person's genotype becomes ./. if that person has DP < 10 or GQ < 20
bcftools filter -S . -e 'FMT/DP<10 | FMT/GQ<20' \
    results/vcf/trio.calls.sitefilt.vcf.gz -Oz -o results/vcf/trio.calls.filtered.vcf.gz \
    2>> results/logs/warnings.log
bcftools index -f -t results/vcf/trio.calls.filtered.vcf.gz
echo "FILTER column:"; bcftools query -f '%FILTER\n' results/vcf/trio.calls.filtered.vcf.gz 2>> results/logs/warnings.log | sort | uniq -c

In [ ]:
FILTERED = "results/vcf/trio.calls.filtered.vcf.gz"
LOWCOV_HET = 5_291_568      # see data/truth/planted_events.tsv (LOWCOV_HET): a TRUE variant in S03
n_pass, failed, missing = 0, [], Counter()
with pysam.VariantFile(FILTERED) as vf:
    for r in vf:
        flt = list(r.filter)
        if flt == ["PASS"]:
            n_pass += 1
        else:
            failed.append((r.pos, r.ref, r.alts[0], flt, round(r.qual, 1), r.info["DP"]))
        for s in SAMPLES:
            if None in r.samples[s]["GT"]:
                missing[s] += 1
print("records:", n_norm, "| PASS:", n_pass, "| failed site filters:", failed)
print("genotypes set to missing per person:", dict(missing))
with pysam.VariantFile(NORM) as vf:
    before_f = next(vf.fetch("chr11", LOWCOV_HET - 1, LOWCOV_HET))
with pysam.VariantFile(FILTERED) as vf:
    after_f = next(vf.fetch("chr11", LOWCOV_HET - 1, LOWCOV_HET))
smp = before_f.samples["S03"]
print(f"chr11:{LOWCOV_HET:,} {before_f.ref}>{before_f.alts[0]}: S03 GT {'/'.join(map(str, smp['GT']))}, "
      f"DP {smp['DP']}, AD {smp['AD']}, GQ {smp['GQ']}, QUAL {before_f.qual:.1f}, INFO/DP {before_f.info['DP']}"
      f" -> FILTER {','.join(after_f.filter)}, GT after filtering {after_f.samples['S03']['GT']}")

In [ ]:
# Check: one site fails, and it is the planted low-coverage variant
assert len(failed) == 1 and failed[0][0] == LOWCOV_HET
assert None in after_f.samples["S03"]["GT"]
RESULTS["record_counts"]["PASS"] = n_pass
RESULTS["filtering"] = {"site_filter": "QUAL<30 || INFO/DP<30 -> FILTER=LowQual (soft)",
                        "genotype_filter": "FMT/DP<10 | FMT/GQ<20 -> GT=./.",
                        "records": n_norm, "PASS": n_pass,
                        "failed_sites": [{"pos": f[0], "ref": f[1], "alt": f[2], "filter": f[3],
                                          "QUAL": f[4], "INFO_DP": f[5]} for f in failed],
                        "genotypes_set_missing": dict(missing)}
RESULTS["low_coverage_true_variant"] = {
    "pos": LOWCOV_HET, "ref": before_f.ref, "alt": before_f.alts[0], "carrier": "S03", "truth_GT": "0/1",
    "called_GT": "/".join(map(str, smp["GT"])), "FMT_DP": smp["DP"], "AD": list(smp["AD"]), "GQ": smp["GQ"],
    "QUAL": round(before_f.qual, 2), "INFO_DP": before_f.info["DP"],
    "FILTER_after": list(after_f.filter), "GT_after_filters": "./.", "removed_by_filters": True}

**What changed?** No record was deleted, but the FILTER column is now filled (`PASS` or `LowQual`) and three genotypes became `./.`. The only site that failed is a **true** heterozygous variant of the mother in the low-coverage interval: the caller found it (QUAL ≈ 95), but with only 8 reads it fails both depth filters.

Filters trade **recall** (finding all true variants) for **precision** (avoiding false ones). Here they cost one true variant and saved nothing, because the main pipeline made no false calls – but in Step 9 you will see call sets where filters matter.

**Your turn.** The filtered site is real. Name two ways to avoid losing variants like this one.
<details><summary>Answer</summary>
Sequence deeper (or improve the lab protocol for this region); use a depth threshold that fits the data instead of a fixed one; look at the reads in IGV; or genotype the region again with family information. Any change of thresholds must be checked on a truth set, because it may also let false calls through.
</details>

## Step 9 · Benchmark the calls against the truth

**Why this step.** Because our data are synthetic, we **know** every true variant (`data/truth/truth_trio.vcf.gz`). For each person we compare called and true variants:
* **TP** (true positive) – a true variant that was called;
* **FP** (false positive) – a call that is not a true variant;
* **FN** (false negative) – a true variant that was missed (or filtered away);
* **precision** = TP / (TP + FP) – how many calls are right; **recall** = TP / (TP + FN) – how many true variants were found; **F1** = their harmonic mean;
* **genotype concordance** – among the TPs, the fraction with the correct genotype (0/1 vs 1/1).

We compare four call sets: the **main pipeline** (Step 7), the same **without duplicate marking**, the same **without a base-quality minimum** (`-Q 0`), and the **main pipeline after the filters** of Step 8. First we make the two extra call sets.

In [ ]:
%%bash
REF=data/ref/chr11_GRCh38_betaglobin.fa.gz
# scenario: no duplicate marking - use the sorted BAM files, in which no read is flagged as duplicate
bcftools mpileup -f $REF -r chr11:5200001-5300000 -Q 13 -a AD,DP -Ou \
    results/bam/S01.sorted.bam results/bam/S02.sorted.bam results/bam/S03.sorted.bam 2>> results/logs/warnings.log \
  | bcftools call -m -v -a GQ --ploidy GRCh38 -Ou 2>> results/logs/warnings.log \
  | bcftools norm -f $REF -m -any -Oz -o results/vcf/scenario_no_markdup.vcf.gz 2>> results/logs/warnings.log
# scenario: -Q 0 - keep every base, whatever its quality
bcftools mpileup -f $REF -r chr11:5200001-5300000 -Q 0 -a AD,DP -Ou \
    results/bam/S01.markdup.bam results/bam/S02.markdup.bam results/bam/S03.markdup.bam 2>> results/logs/warnings.log \
  | bcftools call -m -v -a GQ --ploidy GRCh38 -Ou 2>> results/logs/warnings.log \
  | bcftools norm -f $REF -m -any -Oz -o results/vcf/scenario_Q0.vcf.gz 2>> results/logs/warnings.log
for f in results/vcf/scenario_no_markdup.vcf.gz results/vcf/scenario_Q0.vcf.gz; do
  bcftools index -f -t $f
  echo "$f: $(bcftools view -H $f 2>> results/logs/warnings.log | wc -l) records"
done

In [ ]:
def sample_genotypes(vcf, pass_only=False):
    # {person: {(POS, REF, ALT): number of ALT copies}} for every non-reference genotype
    out = {s: {} for s in SAMPLES}
    with pysam.VariantFile(vcf) as vf:
        for r in vf:
            if pass_only and list(r.filter) != ["PASS"]:
                continue
            for s in SAMPLES:
                gt = r.samples[s]["GT"]
                if None in gt:
                    continue                       # missing genotype = not called
                n_alt = sum(a == 1 for a in gt)    # records are biallelic after norm -m -any
                if n_alt:
                    out[s][(r.pos, r.ref, r.alts[0])] = n_alt
    return out

truth = sample_genotypes("data/truth/truth_trio.vcf.gz")
events = pd.read_csv("data/truth/planted_events.tsv", sep="\t")
SCENARIOS = {"main pipeline": (NORM, False),
             "no duplicate marking": ("results/vcf/scenario_no_markdup.vcf.gz", False),
             "no base-quality minimum (-Q 0)": ("results/vcf/scenario_Q0.vcf.gz", False),
             "main pipeline + filters": (FILTERED, True)}
rows, errors = [], []
for scen, (path, pass_only) in SCENARIOS.items():
    calls = sample_genotypes(path, pass_only)
    total = Counter()
    for s in SAMPLES:
        tp = truth[s].keys() & calls[s].keys()
        fp = calls[s].keys() - truth[s].keys()
        fn = truth[s].keys() - calls[s].keys()
        gt_ok = sum(truth[s][k] == calls[s][k] for k in tp)
        rows.append({"scenario": scen, "sample": s, "TP": len(tp), "FP": len(fp), "FN": len(fn), "GT_match": gt_ok})
        total.update(TP=len(tp), FP=len(fp), FN=len(fn), GT_match=gt_ok)
        for kind, keyset in (("FP", fp), ("FN", fn)):
            for k in sorted(keyset):
                ev = events.loc[events.pos == k[0], "event_id"]
                errors.append({"scenario": scen, "sample": s, "type": kind, "pos": k[0], "ref": k[1], "alt": k[2],
                               "planted_event": ev.iloc[0] if len(ev) else "none"})
    rows.append({"scenario": scen, "sample": "all", **total})
bench = pd.DataFrame(rows)
bench["precision"] = (bench.TP / (bench.TP + bench.FP)).round(3)
bench["recall"] = (bench.TP / (bench.TP + bench.FN)).round(3)
bench["F1"] = (2 * bench.precision * bench.recall / (bench.precision + bench.recall)).round(3)
bench["GT concordance"] = (bench.GT_match / bench.TP).round(3)
bench.to_csv("results/benchmark/benchmark_table.tsv", sep="\t", index=False)
print(bench[bench["sample"] == "all"].drop(columns=["sample", "GT_match"]).to_string(index=False))
print("\nerrors:")
print(pd.DataFrame(errors).to_string(index=False))

In [ ]:
pooled = bench[bench["sample"] == "all"].set_index("scenario")
fig, ax = plt.subplots(figsize=(6, 2.8))
yy = np.arange(len(pooled))
ax.scatter(pooled.precision, yy + 0.12, label="precision")
ax.scatter(pooled.recall, yy - 0.12, label="recall")
ax.set_yticks(yy, pooled.index); ax.invert_yaxis()
ax.set_xlim(0.95, 1.005); ax.set_xlabel("pooled over the three people")
ax.legend(loc="lower left")
plt.show()

In [ ]:
# Check: the main pipeline is perfect; each change shows exactly the planted problem
err = pd.DataFrame(errors)
assert pooled.loc["main pipeline", ["FP", "FN"]].sum() == 0
assert set(err.loc[err.scenario == "no duplicate marking", "planted_event"]) == {"ARTIFACT_JACKPOT"}
assert set(err.loc[err.scenario == "no base-quality minimum (-Q 0)", "planted_event"]) == {"ARTIFACT_LOWBQ"}
assert set(err.loc[err.scenario == "main pipeline + filters", "planted_event"]) == {"LOWCOV_HET"}
assert (bench["GT concordance"] == 1).all()
RESULTS["benchmark"] = bench.to_dict(orient="records")
RESULTS["benchmark_errors"] = errors

**What changed?** We turned four VCF files into a small table of TP / FP / FN per person. The main pipeline found all 33 true variant sites with the right genotypes and no false calls. Each change shows one planted problem:
* **no duplicate marking** → a false 0/1 call in S03 at chr11:5,214,982 (the PCR jackpot of Step 3) – precision drops;
* **no base-quality minimum** (`-Q 0`) → a false 0/1 call in S02 at chr11:5,257,081 (next cell) – precision drops. bcftools' own default (`-Q 1`) gives the same false call, which is why the main pipeline sets `-Q 13`;
* **filters** → the true low-coverage variant of S03 is lost – recall drops.

**Real benchmarks.** For real data, public **truth sets** exist: the *Genome in a Bottle* (GIAB) consortium provides high-confidence variants for reference people such as HG002 and his parents HG003/HG004 (a trio!). Tools such as **hap.py** and **RTG vcfeval** compare call sets *by haplotype*, so different spellings of the same variant still match, and they report results separately for difficult regions (homopolymers, segmental duplications, low-complexity sequence).

In [ ]:
# A closer look at the -Q 0 false call: the same wrong base in many reads, always with quality 12 ('-')
LOWBQ = 5_257_081           # see data/truth/planted_events.tsv (ARTIFACT_LOWBQ)
for q in (13, 0):
    c = base_counts("results/bam/S02.markdup.bam", LOWBQ, min_bq=q)
    print(f"S02 at chr11:{LOWBQ:,} counting bases with quality >= {q:>2}:", c)
with pysam.VariantFile("results/vcf/scenario_Q0.vcf.gz") as vf:
    r = next(vf.fetch("chr11", LOWBQ - 1, LOWBQ))
    print(f"-Q 0 call: {r.ref}>{r.alts[0]} QUAL {r.qual:.1f}, S02 GT {r.samples['S02']['GT']}, AD {r.samples['S02']['AD']}")
lowbq = {"sample": "S02", "pos": LOWBQ, "ref": r.ref, "alt": r.alts[0], "base_quality_of_error": 12,
         "counts_Q13": base_counts("results/bam/S02.markdup.bam", LOWBQ, min_bq=13),
         "counts_Q0": base_counts("results/bam/S02.markdup.bam", LOWBQ, min_bq=0),
         "call_with_Q0": {"QUAL": round(r.qual, 2), "GT": "/".join(map(str, r.samples["S02"]["GT"])),
                          "AD": list(r.samples["S02"]["AD"])},
         "called_in_main_pipeline": LOWBQ in [k[0] for k in sample_genotypes(NORM)["S02"]]}
assert not lowbq["called_in_main_pipeline"] and lowbq["counts_Q13"].get("T", 0) == 0
RESULTS["low_bq_artifact"] = lowbq

**Your turn.** The `-Q 0` false call has QUAL ≈ 20 although more than half of S02's reads show a T. Why is the caller so unsure – and which of our filters would have removed this call anyway?
<details><summary>Answer</summary>
Every T has base quality 12 (about 6 % error chance per base), and the likelihood model knows that low-quality bases are often wrong, so many of them add up to only weak evidence. The site filter <code>QUAL&lt;30</code> from Step 8 would mark it <code>LowQual</code>. Two safety nets – the minimum base quality and the QUAL filter – protect against the same problem.
</details>

## Step 10 · Use the family: trio analysis

**Why this step.** A child receives one copy of each chromosome from each parent. So, at every site, one of the child's alleles must be present in the father and the other in the mother. A violation is a **Mendelian error**. It has two kinds of explanations:
* a **de novo** variant – a new mutation in the child, absent from both parents (every person has a few dozen in the whole genome);
* a mistake – a missed or false genotype call, or a sample swap.

`bcftools +mendelian2` checks every site using the family file `data/trio.ped` (child, father, mother). Then we use the inheritance pattern of the disease: for a **recessive** disease we look for sites where the child is **1/1** and both parents are **0/1**.

In [ ]:
%%bash
cat data/trio.ped
echo "--- sample order in the VCF:"
bcftools query -l results/vcf/trio.calls.filtered.vcf.gz 2>> results/logs/warnings.log | paste -sd ' ' -
echo "--- Mendelian check:"
bcftools +mendelian2 results/vcf/trio.calls.filtered.vcf.gz -P data/trio.ped --rules GRCh38 \
    2>> results/logs/warnings.log | grep -E "^(sites_good|sites_merr|sites_missing)"
echo "--- sites with a Mendelian error:"
bcftools +mendelian2 results/vcf/trio.calls.filtered.vcf.gz -P data/trio.ped --rules GRCh38 -m e \
    2>> results/logs/warnings.log \
  | bcftools query -f '%CHROM:%POS %REF>%ALT [ %SAMPLE=%GT (AD %AD)]\n' 2>> results/logs/warnings.log

In [ ]:
# How solid is the de novo candidate? Look at the parents' reads - all of them, even low-quality and duplicates
DENOVO = 5_272_758          # see data/truth/planted_events.tsv (DENOVO)
with pysam.VariantFile(FILTERED) as vf:
    rec = next(vf.fetch("chr11", DENOVO - 1, DENOVO))
dn = {"pos": DENOVO, "ref": rec.ref, "alt": rec.alts[0], "QUAL": round(rec.qual, 1), "FILTER": list(rec.filter)}
for s in SAMPLES:
    smp = rec.samples[s]
    every_read = base_counts(f"results/bam/{s}.sorted.bam", DENOVO, min_bq=0, skip_duplicates=False)
    dn[s] = {"GT": "/".join(map(str, smp["GT"])), "DP": smp["DP"], "AD": list(smp["AD"]), "GQ": smp["GQ"],
             "all_reads_any_quality": every_read}
    print(f"{s} ({ROLE[s]}): GT {dn[s]['GT']}, AD {dn[s]['AD']}, GQ {dn[s]['GQ']} | every read: {every_read}")

In [ ]:
%%bash
# Recessive model: child (GT[0] = S01) homozygous ALT, father (GT[1]) and mother (GT[2]) heterozygous
bcftools view -H -f PASS -i 'GT[0]="AA" && GT[1]="het" && GT[2]="het"' \
    results/vcf/trio.calls.filtered.vcf.gz 2>> results/logs/warnings.log | cut -f 1-6

In [ ]:
# Check: one Mendelian error (the de novo site), and the recessive model finds only HBB c.20A>T
merr = sh("bcftools +mendelian2 results/vcf/trio.calls.filtered.vcf.gz -P data/trio.ped --rules GRCh38 -m e 2>/dev/null"
          " | bcftools query -f '%POS\n' 2>/dev/null").split()
recessive = sh("bcftools view -H -f PASS -i 'GT[0]=\"AA\" && GT[1]=\"het\" && GT[2]=\"het\"' "
               "results/vcf/trio.calls.filtered.vcf.gz 2>/dev/null | cut -f 2").split()
stats = dict(l.split("\t")[:2] for l in sh("bcftools +mendelian2 results/vcf/trio.calls.filtered.vcf.gz "
             "-P data/trio.ped --rules GRCh38 2>/dev/null").splitlines() if l.startswith("sites_"))
print("Mendelian error sites:", merr, "| recessive-model sites:", recessive, "|", stats)
assert merr == [str(DENOVO)] and recessive == [str(HBB_POS)]
assert dn["S02"]["all_reads_any_quality"].get(dn["alt"], 0) == 0 and dn["S03"]["all_reads_any_quality"].get(dn["alt"], 0) == 0
RESULTS["trio"] = {"mendelian2_summary": stats, "mendelian_error_sites": [int(p) for p in merr],
                   "de_novo_candidate": dn, "recessive_model_hits": [int(p) for p in recessive]}

**What changed?** We did not change the VCF; we asked it two questions. Exactly one site breaks Mendel's rules: chr11:5,272,758, where the child is 0/1 and **neither parent shows a single A read** (27 and 21 reads, counting even low-quality bases and duplicates) – a good de novo candidate. And exactly one site fits the recessive model: *HBB* chr11:5,227,002 – the sickle cell variant. (The low-coverage site is not evaluated because its genotypes were set to missing in Step 8.)

**Your turn.** Before reporting a de novo variant to a family, what would you check?
<details><summary>Answer</summary>
(1) Depth and alt reads in both parents (here: 0 alt reads in 27 and 21 reads). (2) The child's evidence: a fraction of variant reads near 50 %, reads on both strands, not only at read ends, good mapping quality – inspect it in IGV. (3) That the samples really are child, father and mother (overall Mendelian error rate, relatedness checks). (4) Confirm with an independent method, e.g. PCR and Sanger sequencing. For real trios, <code>bcftools +trio-dnm3</code> computes a de novo score (it needs <code>mpileup -a AD,QM,SP</code>).
</details>

## Step 11 · Name the variant (HGVS)

**Why this step.** A genomic position (`chr11:5,227,002 T>A`) is precise, but doctors and databases describe variants relative to a **transcript**: the mRNA of the gene, numbered from the first base of the **coding sequence** (CDS). The **HGVS** nomenclature writes:
* `c.20A>T` – coding DNA position 20 changes from A to T (written on the transcript strand),
* `p.(Glu7Val)` – the protein changes glutamic acid 7 into valine; the brackets mean "predicted from DNA, not measured".

We use the **MANE Select** transcript of *HBB*, **NM_000518.5** (= ENST00000335295.4) – the transcript that NCBI and Ensembl agree to use as the default. *HBB* is on the **minus** strand, so position c.1 is at the right-hand (higher) end of the gene on the genome, and we must reverse-complement.

In [ ]:
ann = pd.read_csv("data/annotation/HBB_MANE_NM_000518.5_hg38.tsv", sep="\t", comment="#")
print(ann[["transcript", "strand", "exon", "exon_start", "exon_end", "cds_start", "cds_end", "cds_length"]].to_string(index=False))
TX, STRAND = ann.transcript[0], ann.strand[0]
coding = [(int(a), int(b)) for a, b in ann.dropna(subset=["cds_start"])[["cds_start", "cds_end"]].values]
COMP = str.maketrans("ACGT", "TGCA")

def genomic_to_c(pos):
    # c. number of a genomic position inside the coding exons; works for both strands
    c = 0
    for start, end in coding:                  # coding parts of the exons, in transcript order 5' -> 3'
        if start <= pos <= end:
            return c + ((end - pos) if STRAND == "-" else (pos - start)) + 1
        c += end - start + 1
    raise ValueError(f"{pos} is not in the coding sequence of {TX}")

# the coding sequence, read 5'->3' on the transcript (minus strand: reverse complement of each exon)
if STRAND == "+":
    cds = "".join(fa.fetch("chr11", a - 1, b) for a, b in coding)
else:
    cds = "".join(fa.fetch("chr11", a - 1, b).translate(COMP)[::-1] for a, b in coding)
B = "TCAG"
AA = "FFLLSSSSYY**CC*WLLLLPPPPHHQQRRRRIIIMTTTTNNKKSSRRVVVVAAAADDEEGGGG"      # the standard genetic code
CODON = {x + y + z: AA[16 * i + 4 * j + k] for i, x in enumerate(B) for j, y in enumerate(B) for k, z in enumerate(B)}
THREE = dict(zip("ACDEFGHIKLMNPQRSTVWY*", ["Ala", "Cys", "Asp", "Glu", "Phe", "Gly", "His", "Ile", "Lys", "Leu",
             "Met", "Asn", "Pro", "Gln", "Arg", "Ser", "Thr", "Val", "Trp", "Tyr", "Ter"]))
protein = "".join(CODON[cds[i:i + 3]] for i in range(0, len(cds), 3))
print(f"\nCDS length {len(cds)} bp = {len(cds) // 3} codons | starts {cds[:3]} | ends {cds[-3:]} | protein starts {protein[:10]}")

def hgvs(pos, ref, alt):
    # HGVS c. and p. names for a single-base substitution given on the plus strand
    c = genomic_to_c(pos)
    t_ref, t_alt = (ref, alt) if STRAND == "+" else (ref.translate(COMP), alt.translate(COMP))
    assert cds[c - 1] == t_ref, "the reference base does not match the transcript"
    k, i = (c - 1) // 3, (c - 1) % 3                         # codon number (0-based) and position in codon
    old = cds[3 * k:3 * k + 3]
    new = old[:i] + t_alt + old[i + 1:]
    a, b = THREE[CODON[old]], THREE[CODON[new]]
    p = f"p.({a}{k + 1}=)" if a == b else f"p.({a}{k + 1}{b})"
    return {"genomic": f"chr11:{pos} {ref}>{alt}", "HGVS_g": f"NC_000011.10:g.{pos}{ref}>{alt}",
            "HGVS_c": f"{TX}(HBB):c.{c}{t_ref}>{t_alt}", "HGVS_p": p, "codon_change": f"{old}>{new}"}

names = {"HBB c.20": hgvs(5_227_002, "T", "A"), "HBB c.9": hgvs(5_227_013, "A", "G")}
print(pd.DataFrame(names).T.to_string())

In [ ]:
# Check: we reproduce the published names
assert cds.startswith("ATG") and protein.endswith("*") and len(cds) == 444
assert names["HBB c.20"]["HGVS_c"] == "NM_000518.5(HBB):c.20A>T" and names["HBB c.20"]["HGVS_p"] == "p.(Glu7Val)"
assert names["HBB c.9"]["HGVS_c"] == "NM_000518.5(HBB):c.9T>C" and names["HBB c.9"]["HGVS_p"] == "p.(His3=)"
RESULTS["naming"] = names

**What changed?** We translated genomic coordinates into the transcript's language: chr11:5,227,002 is the 20th coding base of *HBB*. The plus-strand change T>A is an A>T change on the transcript, which turns codon 7 from GAG (Glu) into GTG (Val): **NM_000518.5(HBB):c.20A>T p.(Glu7Val)**. The second coding variant, c.9T>C, changes CAT to CAC – both code for histidine, so the protein is unchanged: **p.(His3=)** (synonymous).

*Why "Glu7" when textbooks say "HbS = Glu6Val"?* HGVS counts the first methionine (Met1) of the protein. The traditional haemoglobin numbering starts after it, because Met1 is removed from mature β-globin. Same variant, different numbering.

### Optional (needs internet): what do annotation tools say?
The **Ensembl Variant Effect Predictor (VEP)** annotates a variant with its consequence and with predictions such as **CADD** (a general deleteriousness score; higher = more likely harmful), **AlphaMissense** (predicted pathogenicity of amino-acid changes, 0–1) and **SpliceAI** (predicted effect on splicing; delta scores 0–1). If you are offline the cell just prints a message.

In [ ]:
import requests
VEP_URL = ("https://rest.ensembl.org/vep/human/region/11:5227002-5227002:1/A"
           "?content-type=application/json&AlphaMissense=1&CADD=1&SpliceAI=1&mane=1&hgvs=1")
try:
    for attempt in range(3):                     # the public server sometimes answers 500: try again
        resp = requests.get(VEP_URL, timeout=30)
        if resp.status_code == 200:
            break
        time.sleep(3)
    resp.raise_for_status()
    hit = resp.json()[0]
    tc = [t for t in hit["transcript_consequences"] if t.get("mane_select") == "NM_000518.5"][0]
    release = requests.get("https://rest.ensembl.org/info/software?content-type=application/json", timeout=15).json()
    rest = requests.get("https://rest.ensembl.org/info/rest?content-type=application/json", timeout=15).json()
    vep = {"retrieved": True, "date_utc": time.strftime("%Y-%m-%d", time.gmtime()), "url": VEP_URL,
           "ensembl_release": release.get("release"), "rest_api_version": rest.get("release"),
           "assembly": hit.get("assembly_name"), "most_severe_consequence": hit.get("most_severe_consequence"),
           "transcript": tc.get("transcript_id"), "mane_select": tc.get("mane_select"),
           "consequence_terms": tc.get("consequence_terms"), "codons": tc.get("codons"),
           "amino_acids": tc.get("amino_acids"), "HGVSc": tc.get("hgvsc"), "HGVSp": tc.get("hgvsp"),
           "CADD_phred": tc.get("cadd_phred"), "AlphaMissense": tc.get("alphamissense"),
           "SpliceAI": tc.get("spliceai"), "SIFT": [tc.get("sift_prediction"), tc.get("sift_score")],
           "colocated_ids": [c.get("id") for c in hit.get("colocated_variants", [])]}
    for key in ["ensembl_release", "most_severe_consequence", "codons", "HGVSc", "HGVSp", "CADD_phred",
                "AlphaMissense", "SpliceAI", "colocated_ids"]:
        print(f"{key:>24}: {vep[key]}")
except Exception as err:
    vep = {"retrieved": False, "url": VEP_URL, "error": repr(err)}
    print("Could not reach the Ensembl REST API (offline?). Skip this cell. Error:", err)
RESULTS["vep_rest"] = vep

**Discussion: a "likely benign" prediction for a famous disease variant.** Run on a recent Ensembl release, VEP reports `missense_variant` (codons gAg/gTg – the capital letter is the changed base), a CADD score around 14 and an AlphaMissense score around 0.22, classified **likely benign**. Yet HbS is one of the best-established **pathogenic** variants in medicine (ClinVar lists rs334 as pathogenic for sickle cell anaemia). The predictor is not "wrong about biology" – it answers a different question. Missense predictors learn which amino-acid changes tend to **damage the protein** (disturb folding, stability or an active site). HbS does not destabilise β-globin. Instead, the new valine creates a sticky, water-repelling patch on the surface: when oxygen is released, HbS molecules stick together and **polymerise** into long fibres that deform red cells into sickles. Prediction scores are **one line of evidence**; clinical classification (ACMG/AMP guidelines) combines them with population, segregation, functional and clinical data.

**Your turn.** Why is the variant written `T>A` in our VCF but `c.20A>T` in its clinical name?
<details><summary>Answer</summary>
VCF always uses the plus strand of the reference genome. HGVS c. names use the transcript strand. <i>HBB</i> is on the minus strand, so every base is complemented: T on the plus strand is A on the transcript, and the new A on the plus strand is T on the transcript.
</details>

## Step 12 · Wrap-up

| step | tool | what changes | how to verify |
|---|---|---|---|
| 1 inputs & QC | `samtools faidx`, `fastp` | nothing (we look and measure) | reference base T at chr11:5,227,002; R1/R2 counts equal; Q30 ≈ 89 % |
| 2 align | `minimap2 -ax sr`, `samtools sort/index` | FASTQ → sorted BAM with positions and read groups | `flagstat`: ~100 % mapped, properly paired; `@RG SM:` in the header |
| 3 duplicates | `samtools collate/fixmate/sort/markdup` | ~4 % of reads get FLAG 1024 | duplicate % in `flagstat`; jackpot 15 → 1 read |
| 4 coverage | `samtools depth` | depth table | mean ≈ 30×; the planted interval ≈ 8× |
| 5 pileup & GL | `samtools mpileup`, Python | nothing (counts → likelihoods) | hand PL and bcftools PL both pick 0/1 for the father |
| 6 call | `bcftools mpileup` → `bcftools call -mv` | BAM → VCF (sites × people) | HBB: S01 1/1, S02 0/1, S03 0/1 |
| 7 normalise | `bcftools norm -f -m -any` | left-aligned, split records (33 → 34) | normalised calls = normalised truth |
| 8 filter | `bcftools filter -s / -S .` | FILTER column, some GT → `./.` | 1 LowQual site = the low-coverage true variant |
| 9 benchmark | Python (real data: hap.py / vcfeval + GIAB) | nothing (comparison) | precision, recall, F1 per scenario |
| 10 trio | `bcftools +mendelian2`, `bcftools view -i` | nothing (questions) | 1 Mendelian error (de novo), 1 recessive hit (HBB) |
| 11 name | Python + MANE table (+ VEP) | genomic → c./p. names | c.20A>T p.(Glu7Val), c.9T>C p.(His3=) |

### Exit questions
1. Why is the sickle cell variant `T>A` in the VCF but `c.20A>T` in the clinical report?
<details><summary>Answer</summary><i>HBB</i> is on the minus strand. VCF uses the plus strand; HGVS c. uses the transcript strand, so both bases are complemented.</details>

2. A colleague skips duplicate marking "to save time". What could go wrong, and how would you notice?
<details><summary>Answer</summary>A PCR error copied many times looks like a real heterozygous variant (our S03 jackpot became a false 0/1 call with QUAL ≈ 86). Clues: many reads with identical start and end positions, a duplicate rate far above normal, all variant reads at the same read position.</details>

3. Our filters removed one true variant and no false one. Does that mean filtering is useless?
<details><summary>Answer</summary>No. Filters trade recall for precision. On this clean call set they only cost recall, but in the other scenarios the QUAL filter would have removed the low-base-quality false call. Thresholds must be chosen – and checked on a truth set – for the data at hand.</details>

4. Why must both the calls and the truth set be normalised before you compare them?
<details><summary>Answer</summary>The same variant can be written in several ways (left- or right-shifted indels, combined or split multi-allelic records). A naive comparison counts different spellings as different variants and invents false positives and false negatives.</details>

5. AlphaMissense calls HbS "likely benign". Should we doubt the diagnosis?
<details><summary>Answer</summary>No. HbS causes disease by making haemoglobin polymerise when it releases oxygen, not by destabilising the protein, which is what missense predictors mostly learn to recognise. Predictions are one line of evidence; the clinical classification of rs334 as pathogenic rests on decades of genetic, biochemical and clinical evidence.</details>

In [ ]:
# Save all key numbers (the slides and the instructor notes use this file)
RESULTS["runtime_seconds_until_here"] = round(time.time() - T_START, 1)
RESULTS["synthetic_data_note"] = "All reads and genotypes are synthetic (seed 20261015); see data/provenance/."

def to_builtin(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, (tuple, set)):
        return list(o)
    return str(o)

with open("results/p1_results.json", "w") as fh:
    json.dump(RESULTS, fh, indent=1, default=to_builtin)
print("saved results/p1_results.json with", len(RESULTS), "sections; total runtime",
      RESULTS["runtime_seconds_until_here"], "s")

## Optional extension (not run in class)

### Other variant callers
`bcftools` is fast and transparent. Two widely used alternatives re-assemble the reads locally (GATK) or use a deep neural network (DeepVariant). Commands for our files (they need extra installations, so they are shown, not run):

```bash
# GATK HaplotypeCaller: per-person GVCF, then joint genotyping of the family
bgzip -dc data/ref/chr11_GRCh38_betaglobin.fa.gz > results/ref.fa && samtools faidx results/ref.fa
gatk CreateSequenceDictionary -R results/ref.fa
for S in S01 S02 S03; do
  gatk HaplotypeCaller -R results/ref.fa -I results/bam/${S}.markdup.bam \
       -L chr11:5200001-5300000 -ERC GVCF -O results/gatk/${S}.g.vcf.gz
done
gatk CombineGVCFs -R results/ref.fa -V results/gatk/S01.g.vcf.gz -V results/gatk/S02.g.vcf.gz \
     -V results/gatk/S03.g.vcf.gz -O results/gatk/trio.g.vcf.gz
gatk GenotypeGVCFs -R results/ref.fa -V results/gatk/trio.g.vcf.gz -O results/gatk/trio.gatk.vcf.gz

# DeepVariant (Docker); for families there is also DeepTrio (run_deeptrio)
docker run -v "$PWD":/work -w /work google/deepvariant:1.6.1 /opt/deepvariant/bin/run_deepvariant \
    --model_type=WGS --ref=results/ref.fa --reads=results/bam/S01.markdup.bam \
    --regions=chr11:5200001-5300000 --output_vcf=results/deepvariant/S01.vcf.gz --num_shards=2
```
Check the GATK and DeepVariant documentation for the current versions. Normalise their output with `bcftools norm` and benchmark it with the code of Step 9.

### Look at the reads in IGV
1. Start IGV (desktop). Genome: *Genomes → Load Genome from File…* → `data/ref/chr11_GRCh38_betaglobin.fa.gz` (the `.fai` and `.gzi` files must sit next to it). Because the coordinates are real GRCh38, you can also choose IGV's built-in **Human (GRCh38/hg38)** genome and see the RefSeq genes.
2. *File → Load from File…* → `results/bam/S01.markdup.bam`, `S02…`, `S03…` (with their `.bai`) and `results/vcf/trio.calls.filtered.vcf.gz` (with `.tbi`).
3. Go to `chr11:5,226,980-5,227,030`: the child shows A in every read at 5,227,002. In the father, the reads with A also carry G at 5,227,013 (c.9T>C): the two variants are on the **same** chromosome copy (*in cis*).
4. Load `results/bam/S03.sorted.bam` and go to `chr11:5,214,982` to see the jackpot stack (identical start and end). IGV hides duplicate-flagged reads by default, which is why the markdup BAM shows only one.
5. Go to `chr11:5,257,081` in S02 and colour/shade bases by quality: the T's are faint (quality 12). Go to `chr11:5,291,568` to see how few reads remain in the low-coverage interval.